In [1]:
from pathlib import Path
from dotenv import load_dotenv
import dspy
import config

# Import our custom RAG system components
from src.rag_system import (
    PPLDataStores, 
    AgenticRAG, 
    PPLTools, 
    get_final_agentic_tools,
)

load_dotenv()

# --- Configure LLMs ---
lm = dspy.LM(config.RAG_LM_MODEL, max_tokens=8000)
dspy.configure(lm=lm)

# --- Load Data Stores ---
CONFIG = {
    "EMBEDDING_MODEL": config.EMBEDDING_MODEL,
    "AGENTIC_DB_PATH": config.AGENTIC_DB_PATH,
    "TEXTSPLIT_DB_PATH": config.TEXTSPLIT_DB_PATH,
    "METADATA_DB_PATH": config.METADATA_DB_PATH,
    "AGENTIC_BM25_PATH": config.AGENTIC_BM25_PATH,
    "TEXTSPLIT_BM25_PATH": config.TEXTSPLIT_BM25_PATH,
    "AGENTIC_COLLECTION": config.AGENTIC_COLLECTION,
    "TEXTSPLIT_COLLECTION": config.TEXTSPLIT_COLLECTION,
    "METADATA_COLLECTION": config.METADATA_COLLECTION,
}
data_stores = PPLDataStores(config=CONFIG)

# --- Initialize Agentic RAG components ---
agentic_tool_functions = PPLTools(
    vectorstore=data_stores.agentic_vectorstore,
    metastore=data_stores.metadata_vectorstore,
    bm25_retriever=data_stores.agentic_bm25_retriever,
    query_lm=lm,
    corpus_file_path=config.CORPUS_FILE
)
agentic_tools = get_final_agentic_tools(agentic_tool_functions)
retriever_path = Path("../results/optimized_programs/compiled_retriever_bootstrap_v4.json")

# --- Create AgenticRAG system ---
agentic_rag = AgenticRAG(tools=agentic_tools, retriever_path=retriever_path)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x71f1d724e7d0>>
Traceback (most recent call last):
  File "/home/webui/Project/venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


--- Initializing PPL Data Stores ---
  -> Loading collection 'PPL_agentic_documents' from /home/webui/Project/data/stores/chroma_agentic_db...
  -> Loading collection 'PPL_text_split_documents' from /home/webui/Project/data/stores/chroma_text_split_db...
  -> Loading collection 'PPL_document_metadata' from /home/webui/Project/data/stores/chroma_metadata_db...
  -> Loading BM25 retriever from /home/webui/Project/data/stores/bm25_agentic_retriever.pkl...
  -> Loading BM25 retriever from /home/webui/Project/data/stores/bm25_textsplit_retriever.pkl...
✅ All data stores loaded successfully.
✅ PPLTools initialized with all necessary data stores.
[AgenticRAG] ✅ Loading compiled retriever state from: /home/webui/Project/results/optimized_programs/compiled_retriever_bootstrap_v4.json


In [ ]:
question = "If I accidentally spill some radioactive tracer and arsenic-containing dust on my arm, what's the first aid process? Do I just wash it? And who do I tell to make sure I get the right health follow-up?"

retriever = agentic_rag.retriever
trace = retriever(question=question)

In [4]:
print(trace.titles)

['Working Safely with Arsenic Guideline', 'Emergency Response Plan for Radioactive Liquid Spills Procedure']


In [ ]:
print(dspy.inspect_history(n=1))





[2025-10-24T02:26:23.772969]

System message:

Your input fields are:
1. `question` (str): The user's original question.
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `titles` (list[str]): A list of source document titles relevent to answering the question.
3. `notes` (str): Record any important findings or nuances about your observations that are not captured in the final list of documents.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## titles ## ]]
{titles}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## notes ## ]]
{notes}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Conduct research to find all policy document titles relevant to answering the user's question.
        
        **Firs

In [ ]:
question="I've lost my university laptop. Do I need to report this to anyone?"
ans = "Yes, you need to report the loss of your university laptop to IT support immediately. Both the Cyber Security Policy and the Cyber Security Incident Response Procedure state that consumers (including students and staff) are responsible for reporting potential cyber security incidents, including accidental incidents such as a lost laptop or device, to IT support."

question="I just accidentally spilled some radioactive liquid in my lab while working on my research. I'm not injured, but I'm worried about contamination. What are the immediate steps I need to take right now?"
ans = 'Here are the immediate steps you should take following the radioactive liquid spill, based on the Emergency Response Plan for Radioactive Liquid Spills Procedure:\n\n1.  **Treatment of Injured Person:** If you have any skin-penetrating injuries, prioritize treatment according to established first aid procedures. If the injury is severe, call emergency services at 000, then UQ Security at 3365 3333, and describe the situation. Take a copy of the [Radiation Safety Data Sheet](https://policies.uq.edu.au/document/view-current.php?id=432) with you to the medical practitioner. If you have been contaminated, remove contaminated clothing and decontaminate any affected skin areas (see clauses 17-20 of the Emergency Response Plan for Radioactive Liquid Spills Procedure).\n2.  **Notify the Radiation Safety Officer (RSO) and Other Persons Working in the Area of the Spill:** Notify your local RSO as soon as possible. The RSO will assess the incident and determine the necessary response. You can find the local RSO contact details on the Health, Safety and Wellness webpage: [UQ Safety Network Contacts](https://policies.uq.edu.au/download.php?id=440&version=1&associated). Also, provide the RSO with the [Radioisotope Fact Sheet](https://policies.uq.edu.au/document/view-current.php?id=432) for the spilled material before any cleanup begins.\n3.  **Contain the Spill and Monitor for Contamination:** Put on all appropriate personal protective equipment (PPE) before attempting to clean up the spill. Drop absorbent paper or pads onto the spill and wipe up the remaining material, working inward towards the center of the spill to prevent further spread. After containing the spill, determine the extent of any remaining contamination using a radiation survey meter appropriate for the contaminant.\n4.  **Decontamination:** Decontamination should be carried out in a planned and logical manner. Assistance from the local RSO is available if required. Laboratories in which radioactive materials are used must maintain a decontamination kit so that the necessary materials are always at hand.\n\nAfter these immediate actions, the RSO will investigate the incident and report it in [UQSafe](https://policies.uq.edu.au/download.php?id=591&version=2&associated).'

response = agentic_rag(question="what is the definition of delegate?")
print(response.answer)

Here are the immediate steps you should take following the radioactive liquid spill, based on the Emergency Response Plan for Radioactive Liquid Spills Procedure:

1.  **Treatment of Injured Person:** If you have any skin-penetrating injuries, prioritize treatment according to established first aid procedures. If the injury is severe, call emergency services at 000, then UQ Security at 3365 3333, and describe the situation. Take a copy of the [Radiation Safety Data Sheet](https://policies.uq.edu.au/document/view-current.php?id=432) with you to the medical practitioner. If you have been contaminated, remove contaminated clothing and decontaminate any affected skin areas (see clauses 17-20 of the Emergency Response Plan for Radioactive Liquid Spills Procedure).
2.  **Notify the Radiation Safety Officer (RSO) and Other Persons Working in the Area of the Spill:** Notify your local RSO as soon as possible. The RSO will assess the incident and determine the necessary response. You can find the local RSO contact details on the Health, Safety and Wellness webpage: [UQ Safety Network Contacts](https://policies.uq.edu.au/download.php?id=440&version=1&associated). Also, provide the RSO with the [Radioisotope Fact Sheet](https://policies.uq.edu.au/document/view-current.php?id=432) for the spilled material before any cleanup begins.
3.  **Contain the Spill and Monitor for Contamination:** Put on all appropriate personal protective equipment (PPE) before attempting to clean up the spill. Drop absorbent paper or pads onto the spill and wipe up the remaining material, working inward towards the center of the spill to prevent further spread. After containing the spill, determine the extent of any remaining contamination using a radiation survey meter appropriate for the contaminant.
4.  **Decontamination:** Decontamination should be carried out in a planned and logical manner. Assistance from the local RSO is available if required. Laboratories in which radioactive materials are used must maintain a decontamination kit so that the necessary materials are always at hand.

After these immediate actions, the RSO will investigate the incident and report it in [UQSafe](https://policies.uq.edu.au/download.php?id=591&version=2&associated).